###  Script to generate a parameter file for the PA MRV simulations

The base parameter file is API39. 
We use FatesPFTIndexSwapper.py with –pft-indices=6,6  to  get fates_params_PA_default_2pfts.nc

MAKE SURE TO MANUALLY INCREASE THE NUMBER OF SIZE BINS. 

ALSO ADD IN THE FATES REFORESTATION PARAMETERS (they should already be there  if the  parameter 
file is copied from the MRV branch but check!)

### Libraries

In [1]:
from scipy.stats import qmc
import numpy as np
import xarray as xr
import csv
import pandas as pd
import os
import netCDF4 as nc4
from scipy.io import netcdf as nc
import sys
from tempfile import TemporaryFile                                                                                                                                 
import argparse                                                                                                                                                     
import shutil                                                                                                                                                       
import tempfile 
import random

import math as m
from scipy.optimize import curve_fit
from scipy.stats import linregress

nan = float("nan")

import modp as mp

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import cartopy.crs as ccrs
from matplotlib import cm

dark = cm.get_cmap('Paired', 12)
colormap = dark(range(12))

/global/homes/j/jneedham/.conda/envs/myenv/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
ERROR 1: PROJ: proj_create_from_database: Open of /global/homes/j/jneedham/.conda/envs/myenv/share/proj failed


### Load allometry parameters 
From FATES-MRV-allometry.ipynb

In [2]:
bldex_ps = pd.read_csv('/global/homes/j/jneedham/FATES-MRV/allom_scripts/pft6_params.csv')
poplar_ps = pd.read_csv('/global/homes/j/jneedham/FATES-MRV/allom_scripts/poplar_params.csv')

bldex_ps = bldex_ps.set_index('pft')

In [3]:
bldex_ps

,d2h1,d2h2,d2h3,d2ca,d2bl2,wood_density,agb1,agb2
pft,,,,,,,,
bldex,46.696808,0.926756,28.944679,0.436977,1.365634,0.582007,0.129183,0.981921
nleex,52.697076,0.948560,46.792526,0.201809,1.463887,0.371920,0.081875,1.020661


In [4]:
poplar_ps

,Unnamed: 0,d2h1,d2h2,d2h3,wood_density,d2ca,d2bl2,agb1,agb2
0,0,30.846537,1.105258,18.057759,0.471202,0.334165,1.312636,0.129183,0.981921


## Natural two PFT parameter file

In [5]:
# starting parameter file
input_fname  = '/global/homes/j/jneedham/FATES-MRV/param_files/v9/fates_params_PA_default_2pfts.nc'
# final parameter file name
fout = '/global/homes/j/jneedham/FATES-MRV/param_files/v9/fates_params_PA_nat.nc'
    
shutil.copyfile(input_fname, fout)  

organ = 1

### Global parameters ###
pft = 0
var = 'fates_mort_disturb_frac'
val = 1.0
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_mort_understorey_death'
val = 1.0
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_maintresp_nonleaf_baserate'
val = 2.0e-06
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)


### Allometry parameters ###
for i in range(1,3):
    # height
    var = 'fates_allom_d2h1'
    val = bldex_ps.d2h1['bldex']
    mp.main(var = var, pft = i, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

    var = 'fates_allom_d2h2'
    val = bldex_ps['d2h2']['bldex']
    mp.main(var = var, pft = i, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

    var = 'fates_allom_d2h3'
    val = bldex_ps['d2h3']['bldex']
    mp.main(var = var, pft = i, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

    # crown area
    var = 'fates_allom_d2ca_coefficient_max'
    val = bldex_ps['d2ca']['bldex']
    mp.main(var = var, pft = i, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

    var = 'fates_allom_d2ca_coefficient_min'
    val = bldex_ps['d2ca']['bldex']
    mp.main(var = var, pft = i, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

    var = 'fates_allom_d2bl2'
    val = bldex_ps['d2bl2']['bldex']
    mp.main(var = var, pft = i, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

    var =  'fates_allom_blca_expnt_diff'
    val = 0
    mp.main(var = var, pft = i, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

    # Wood density
    var =  'fates_wood_density'
    val = bldex_ps['wood_density']['bldex']
    mp.main(var = var, pft = i, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

    # AGB
    var =  'fates_allom_agb1'
    val = bldex_ps['agb1']['bldex']
    mp.main(var = var, pft = i, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

    var =  'fates_allom_agb2'
    val = bldex_ps['agb2']['bldex']
    mp.main(var = var, pft = i, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)



    ### Other Parameters that are the same for both PFTs ###

     # increase hydraulic failure mortality
    var = 'fates_mort_hf_sm_threshold'
    val = 0.3
    mp.main(var = var, pft = i, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)  

    
### Parameters that differ between PFTs
var = 'fates_alloc_storage_cushion'
pft = 1
val = 1.4
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 1.8
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_leaf_vcmax25top'
pft = 1
val = 68
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 60
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_leaf_slatop'
pft = 1
val = 0.14
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 0.08
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_maintresp_leaf_atkin2017_baserate'
pft = 1
val = 1.756
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 1.756
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_maintresp_leaf_vert_scaler_coeff2'
pft = 1
val = 2.2
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 2.1
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_grperc'
pft = 1
val = 0.05
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

pft = 2
val = 0.03
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_mort_bmort'
pft = 1
val = 0.021
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 0.019
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_mort_scalar_cstarvation'
pft = 1
val = 0.85
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 0.82
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_recruit_seed_alloc'
pft = 1
val = 0.027
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 0.025
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_recruit_seed_alloc_mature'
pft = 1
val = 0.75
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 0.6
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_recruit_seed_dbh_repro_threshold'
pft = 1
val = 50
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 60
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_recruit_seed_germination_rate'
pft = 1
val = 0.0114
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 0.00912
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_recruit_seed_supplement'
pft = 1
val = 0.018
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 0.018
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_turnover_branch'
pft = 1
val = 134
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 161
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)


var = 'fates_recruit_init_density'
pft = 1
val = 0.025
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)
pft = 2
val = 0.025
mp.main(var = var, pft = pft, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)




## Logging file

In [6]:
# starting parameter file
input_fname  = '/global/homes/j/jneedham/FATES-MRV/param_files/v9/fates_params_PA_nat.nc'
# final parameter file name
fout = '/global/homes/j/jneedham/FATES-MRV/param_files/v9/fates_params_PA_logging.nc'
    
shutil.copyfile(input_fname, fout)  

organ = 1

for i in range(1,3):

    var = 'fates_mort_bmort'
    val = 5.0
    mp.main(var = var, pft = i, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

    


var = 'fates_landuse_logging_coll_under_frac'
val = 0.0
mp.main(var = var, pft = 0, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_landuse_logging_collateral_frac'
val = 0.0
mp.main(var = var, pft = 0, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_landuse_logging_dbhmax_infra'
val = 0.0
mp.main(var = var, pft = 0, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_landuse_logging_dbhmin'
val = 0.0
mp.main(var = var, pft = 0, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_landuse_logging_direct_frac'
val = 1.0
mp.main(var = var, pft = 0, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_landuse_logging_event_code'
val = 19500101
mp.main(var = var, pft = 0, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_landuse_logging_export_frac'
val = 1.0
mp.main(var = var, pft = 0, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)

var = 'fates_landuse_logging_mechanical_frac'
val = 0.0
mp.main(var = var, pft = 0, fin=fout, val = val, 
                fout = fout, O = 1, organ = organ)


## Poplar file

In [7]:
poplar_ps

,Unnamed: 0,d2h1,d2h2,d2h3,wood_density,d2ca,d2bl2,agb1,agb2
0,0,30.846537,1.105258,18.057759,0.471202,0.334165,1.312636,0.129183,0.981921


In [8]:
# starting parameter file
input_fname  = '/global/homes/j/jneedham/FATES-MRV/param_files/v9/fates_params_PA_nat.nc'
# final parameter file name
fout = '/global/homes/j/jneedham/FATES-MRV/param_files/v9/fates_params_PA_refor.nc'
    
shutil.copyfile(input_fname, fout)  

organ = 1
pft = 1

# height
var = 'fates_allom_d2h1'
val = poplar_ps['d2h1']
mp.main(var = var, pft = pft, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)

var = 'fates_allom_d2h2'
val = poplar_ps['d2h2']
mp.main(var = var, pft = pft, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)

var = 'fates_allom_d2h3'
val = poplar_ps['d2h3']
mp.main(var = var, pft = pft, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)

# crown area
var = 'fates_allom_d2ca_coefficient_max'
val = poplar_ps['d2ca']
mp.main(var = var, pft = pft, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)

var = 'fates_allom_d2ca_coefficient_min'
val = poplar_ps['d2ca']
mp.main(var = var, pft = pft, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)

var = 'fates_allom_d2bl2'
val = poplar_ps['d2bl2']
mp.main(var = var, pft = pft, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)

var =  'fates_allom_blca_expnt_diff'
val = 0
mp.main(var = var, pft = pft, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)

# Wood density
var =  'fates_wood_density'
val = poplar_ps['wood_density']
mp.main(var = var, pft = pft, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)


### We don't want the other PFT germinating or surviving
pft = 2

var = 'fates_mort_bmort'
val = 5.0
mp.main(var = var, pft = pft, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)

var = 'fates_recruit_seed_germination_rate'
val = 0.0
mp.main(var = var, pft = pft, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)

# no supplemental seed rain in the reforestation plot
var = 'fates_recruit_seed_supplement'
val = 0
mp.main(var = var, pft = 1, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)
mp.main(var = var, pft = 2, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)

# only PFT  1 gets planted
var = 'fates_reforestation_density'
val = 0.09
mp.main(var = var, pft = 1, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)
val = 0.0
mp.main(var = var, pft = 2, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)

var = 'fates_reforestation_height'
val = 0.22
mp.main(var = var, pft = 1, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)

var =  'fates_reforestation_event_code'
val = 2
mp.main(var = var, pft = 0, fin=fout, val = val, 
            fout = fout, O = 1, organ = organ)
